List all collections
=======================
curl http://localhost:6333/collections



In [1]:
import sys, os

# Get the directory where the notebook is running
notebook_dir = os.getcwd()

# Go up three levels: models → src → backend → project root
project_root = os.path.abspath(
    os.path.join(notebook_dir, "..")
)

# Add project root to Python path
if project_root not in sys.path:
    sys.path.append(project_root)

print("Project root:", project_root)
print("src in path:", any("src" in p for p in sys.path))

Project root: c:\Users\azari\OneDrive\Desktop\development\software_projects\teaching_projects\Minimal-RAG-Engine\backend\src
src in path: True


In [ ]:
#Connect to Qdrant
from qdrant_client import QdrantClient, models

qdrant = QdrantClient(
    url="http://localhost:6333",  # your docker container
)


In [3]:
#List collections
qdrant.get_collections()

CollectionsResponse(collections=[CollectionDescription(name='documents')])

In [4]:
#Check if a collection exists
qdrant.collection_exists("documents")

True

In [5]:
#Generate embeddings
import os
from services.rag.document_handler import documentHandler

filename = "Microsoft.txt"
def file_finder(filename, base_dir="files"):
    for root, dirs, files in os.walk(base_dir):
        if filename in files:
            return os.path.join(root, filename)
    return None

result = file_finder(filename)
if result:
    print(f"Found: {result}")
else:
    print("File not found.")

doc = documentHandler(file_path=result, file_name=filename)
text = ["When was microsoft founded"]
embedding = doc.embed_batch(text)
query_vector = embedding[0]

c:\Users\azari\OneDrive\Desktop\development\software_projects\teaching_projects\Minimal-RAG-Engine\backend\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Loading .env from: c:\Users\azari\OneDrive\Desktop\development\software_projects\teaching_projects\Minimal-RAG-Engine\backend\.env
Found: files\Microsoft.txt


Loading weights: 100%|██████████| 391/391 [00:00<00:00, 34239.57it/s]


In [9]:
print(query_vector)

[-0.024341529235243797, 0.0036892376374453306, -0.016225725412368774, 0.026452744379639626, 0.011748666875064373, 0.007807776797562838, -0.027751417830586433, -0.0005534756928682327, 0.07291726022958755, -0.0016613006591796875, 0.017159517854452133, -0.0005009424639865756, -0.0216881912201643, -0.043324064463377, 0.040103815495967865, -0.031007325276732445, 0.04186931997537613, -0.00890359841287136, 0.010052886791527271, -0.043137356638908386, 0.039331741631031036, -0.06649398803710938, 0.04711971804499626, -0.010858599096536636, 0.03385693207383156, 0.024685608223080635, -0.021433399990200996, -0.014568755403161049, 0.05163922533392906, -0.03909523785114288, -0.01003270037472248, -0.03527519851922989, 0.016805410385131836, -0.01385846920311451, -0.0021493546664714813, -0.0028018970042467117, 0.01615910604596138, -0.03479594364762306, -0.06262369453907013, -0.05981270223855972, -0.030664796009659767, 0.028701918199658394, -0.01435523945838213, 0.007146359421312809, -0.04661564156413078

In [14]:
#Vector search
result = qdrant.query_points(
    collection_name="documents",
    query=query_vector,   # just pass the list of floats directly
    limit=5
)

for point in result.points:
    print(point.payload, point.score)

{'text': "Its flagship hardware products are the Surface lineup of PCs and the Xbox brand of video game consoles, the latter including the Xbox network. The company also offers online services such as Bing web search, the MSN web portal, the Outlook.com (Hotmail) email service, and the Microsoft Store. In the enterprise and development fields, Microsoft. most notably provides the Azure cloud computing platform, Microsoft SQL Server database software, and Visual Studio. Microsoft became the third publicly traded U.S. company to be valued at over $1 trillion in April 2019. It has been criticized for monopolistic practices, and the company's software received criticism for problems with ease of use, robustness, and security. Microsoft has also been criticized for its role in providing services to Israel during the Gaza war.\n\n== History ==\n\n=== 1972–1985: Founding ===", 'source': 'Microsoft.txt', 'chunk_index': 2, 'total_chunks': 83} 0.7070621
{'text': "Microsoft Corporation is an Amer

In [17]:
#Filtered search
result = qdrant.query_points(
    collection_name="documents",
    query=query_vector,
    query_filter=models.Filter(
        must=[
            models.FieldCondition(
                key="source",
                match=models.MatchValue(value=filename)
            )
        ]
    ),
    limit=5
)

for point in result.points:
    print(point.payload, point.score)

{'text': "Its flagship hardware products are the Surface lineup of PCs and the Xbox brand of video game consoles, the latter including the Xbox network. The company also offers online services such as Bing web search, the MSN web portal, the Outlook.com (Hotmail) email service, and the Microsoft Store. In the enterprise and development fields, Microsoft. most notably provides the Azure cloud computing platform, Microsoft SQL Server database software, and Visual Studio. Microsoft became the third publicly traded U.S. company to be valued at over $1 trillion in April 2019. It has been criticized for monopolistic practices, and the company's software received criticism for problems with ease of use, robustness, and security. Microsoft has also been criticized for its role in providing services to Israel during the Gaza war.\n\n== History ==\n\n=== 1972–1985: Founding ===", 'source': 'Microsoft.txt', 'chunk_index': 2, 'total_chunks': 83} 0.7070621
{'text': "Microsoft Corporation is an Amer

In [20]:
#retrieve ids
points, next_page = qdrant.scroll(
    collection_name="documents",
    limit=10
)

for p in points:
    print(p.id, p.payload)


05808056-cb51-4208-a415-e2110d54a898 {'text': '=== 1985–1994: Windows and Office ===\n\nMicrosoft released Windows 1.0 on November 20, 1985, as a graphical extension for MS-DOS, despite having begun jointly developing OS/2 with IBM that August. Microsoft moved its headquarters from Bellevue to Redmond, Washington, on February 26, 1986, and went public with an initial public offering (IPO) at the NASDAQ exchange on March 13, with the resulting rise in stock making an estimated four billionaires and 12,000 millionaires from Microsoft employees. Microsoft released its version of OS/2 to original equipment manufacturers (OEMs) on April 2, 1987. In 1990, the Federal Trade Commission examined Microsoft for possible collusion due to the partnership with IBM, marking the beginning of more than a decade of legal clashes with the government. Meanwhile, the company was at work on Microsoft Windows NT, which was heavily based on its copy of the OS/2 code.', 'source': 'Microsoft.txt', 'chunk_index'

In [21]:
#Retrieve a point
id = "05808056-cb51-4208-a415-e2110d54a898"
qdrant.retrieve("documents", ids=[id])


[Record(id='05808056-cb51-4208-a415-e2110d54a898', payload={'text': '=== 1985–1994: Windows and Office ===\n\nMicrosoft released Windows 1.0 on November 20, 1985, as a graphical extension for MS-DOS, despite having begun jointly developing OS/2 with IBM that August. Microsoft moved its headquarters from Bellevue to Redmond, Washington, on February 26, 1986, and went public with an initial public offering (IPO) at the NASDAQ exchange on March 13, with the resulting rise in stock making an estimated four billionaires and 12,000 millionaires from Microsoft employees. Microsoft released its version of OS/2 to original equipment manufacturers (OEMs) on April 2, 1987. In 1990, the Federal Trade Commission examined Microsoft for possible collusion due to the partnership with IBM, marking the beginning of more than a decade of legal clashes with the government. Meanwhile, the company was at work on Microsoft Windows NT, which was heavily based on its copy of the OS/2 code.', 'source': 'Microso

In [ ]:
#Delete points
id = ""
qdrant.delete(
    collection_name="documents",
    points_selector=models.PointIdsList(points=[id])
)


In [ ]:
#Delete a collection
qdrant.delete_collection("documents")
